# Data Loader

In [ ]:
# --- Cell 1: Imports ---
import os
from pathlib import Path
import pandas as pd # Useful later for converting your loaded data into a DataFrame

In [ ]:
# --- Cell 2: TeamHanSum Data Loader ---

class DataLoader:
    """
    A data loader designed to parse the hierarchical directory structure
    of the HanSum project, pairing English translations with their
    corresponding English summaries.
    """
    def __init__(self, base_dir: str):
        self.base_dir = Path(base_dir)
        self.translated_dir = self.base_dir / "translated"
        self.summary_dir = self.base_dir / "summary"
        self.data = []

    def load_data(self) -> list:
        """
        Walks through the translated directory to find 'target.txt' files
        and pairs them with 'target.summary.txt' from the summary directory.

        Returns:
            list: A list of dictionaries containing metadata, translation, and summary.
        """
        self.data = []
        if not self.translated_dir.exists():
            print(f"Error: Translated directory not found at {self.translated_dir}")
            return []

        # Use rglob to recursively find all target.txt files
        translation_files = list(self.translated_dir.rglob("target.txt"))

        for trans_file in translation_files:
            # Calculate relative path from the 'translated' folder
            rel_path = trans_file.relative_to(self.translated_dir)

            # Construct the corresponding summary file path
            # target.txt -> target.summary.txt
            summary_rel_path = rel_path.parent / "target.summary.txt"
            summary_file = self.summary_dir / summary_rel_path

            if summary_file.exists():
                with open(trans_file, 'r', encoding='utf-8') as f:
                    translation_text = f.read().strip()

                with open(summary_file, 'r', encoding='utf-8') as f:
                    summary_text = f.read().strip()

                # Extract metadata from path (corpus/level1/level2)
                # rel_path.parts looks like ('史记', '十二本纪', '五帝本纪', 'target.txt')
                parts = rel_path.parts[:-1] # Exclude the filename

                self.data.append({
                    "corpus": parts[0] if len(parts) > 0 else "unknown",
                    "level1": parts[1] if len(parts) > 1 else "unknown",
                    "level2": parts[2] if len(parts) > 2 else "unknown",
                    "translation": translation_text,
                    "summary": summary_text,
                    "rel_path": str(rel_path.parent)
                })
            else:
                # Useful for identifying missing generations in your pipeline
                print(f"Warning: Summary file not found for {trans_file}")

        return self.data

    def get_statistics(self) -> dict:
        """Returns the count of document pairs per corpus."""
        stats = {}
        for item in self.data:
            corpus = item['corpus']
            stats[corpus] = stats.get(corpus, 0) + 1
        return stats

In [ ]:
# --- Cell 3: Execution & Testing ---

# Assuming you cloned the repo in Colab, this is the standard path.
# Change this if you mounted Google Drive instead (e.g., '/content/drive/MyDrive/.../data/processed')
COLAB_BASE_PATH = "/content/drive/MyDrive/github/classical-chinese-summarization/data/processed"

# Instantiate and load
loader = DataLoader(COLAB_BASE_PATH)
samples = loader.load_data()

print(f"✅ Successfully loaded {len(samples)} pairs of translation and summary.")
print("-" * 40)
print("📊 Statistics by corpus:")
for corpus, count in loader.get_statistics().items():
    print(f"   - {corpus}: {count} documents")

# Verify the first sample to ensure text extraction worked correctly
if samples:
    print("-" * 40)
    print("🔍 First Sample Metadata Inspection:")
    print(f"   Corpus: {samples[0]['corpus']}")
    print(f"   Path:   {samples[0]['rel_path']}")
    print(f"\n   Translation Extract:\n   >>> {samples[0]['translation'][:150]}...")
    print(f"\n   Summary Extract:\n   >>> {samples[0]['summary'][:150]}...")

# Summ-eval-Summac_conv

In [ ]:
!pip install summac

In [ ]:
# --- Cell 4: SummaC-Conv Evaluation & Export ---
# Add a force reinstall for summac to address potential dependency issues

import time
import pandas as pd
import nltk
from summac.model_summac import SummaCConv

# 1. Download required NLTK sentence tokenizer (SummaC needs this to split documents)
nltk.download('punkt')
nltk.download('punkt_tab') # Download 'punkt_tab' resource as specified by the error

# 2. Convert the loaded list of dictionaries into a Pandas DataFrame
df = pd.DataFrame(samples)
print(f"Total dataset size: {len(df)} pairs.")

# 3. Create a small test subset of 10 samples
#test_df = df.head(10).copy()
#print(f"Running test on {len(test_df)} samples...")

# 4. Initialize SummaC-Conv (State-of-the-Art for Factual Consistency)
# - models=["vitc"]: Uses the RoBERTa model fine-tuned on VitaminC and MNLI
# - device="cuda": Forces the model onto your Colab GPU for fast processing
print("Initializing SummaCConv with vitc on CUDA (GPU)... This may take a minute to download weights.")
model_conv = SummaCConv(
    models=["vitc-base"],
    bins='percentile',
    granularity="sentence",
    nli_labels="e",
    device="cuda",
    start_file="default",
    agg="mean",
    imager_load=False
)

# 1. Define the directory FIRST
output_dir = "/content/drive/MyDrive/github/classical-chinese-summarization/data/processed/evaluation/summary_eval"

# 2. Tell Python to create this folder if it cannot find it
os.makedirs(output_dir, exist_ok=True)

# 3. Define the actual file path inside that directory
output_csv_path = f"{output_dir}/summac_test_results.csv"

print(f"Results will be safely saved to: {output_csv_path}")


# Track time to estimate full dataset cost
start_time = time.time()

# 6. Iterate through the DataFrame
for index, row in df.iterrows():
    translation = row['translation']
    summary = row['summary']

    # Calculate SummaC score (expects lists of strings)
    result = model_conv.score([translation], [summary])
    score = result["scores"][0]

    # Save score back to the specific row in the DataFrame
    test_df.loc[index, 'summac_conv_score'] = score

    # Iteratively save the DataFrame to Google Drive
    test_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

    # Print progress
    corpus = row['corpus']
    rel_path = row['rel_path']
    print(f"Processed [{index+1}/10] - {corpus}: {rel_path} | Score: {score:.4f}")

end_time = time.time()
elapsed_time = end_time - start_time

print("-" * 40)
print(f"✅ Test Complete! Results saved to {output_csv_path}")
print(f"⏱️ Total time for 10 samples: {elapsed_time:.2f} seconds.")
print(f"📈 Average time per sample: {elapsed_time / 10:.2f} seconds.")

# Projecting the time for your entire dataset:
total_samples = len(df)
projected_seconds = (elapsed_time / 10) * total_samples
print(f"🔮 Estimated time to process all {total_samples} samples: {projected_seconds / 60:.2f} minutes.")

# One-Shot Sample SummaC

In [ ]:
import nltk
from summac.model_summac import SummaCZS, SummaCConv

nltk.download('punkt')
nltk.download('punkt_tab') # Ensure 'punkt_tab' is downloaded before model initialization

model_zs = SummaCZS(granularity="sentence", model_name="vitc-base", device="cpu") # If you have a GPU: switch to: device="cuda"
model_conv = SummaCConv(models=["vitc-base"], bins='percentile', granularity="sentence", nli_labels="e", device="cpu", start_file="default", agg="mean")



document = """


Guan Yu, whose courtesy name is Yunchang, originally called Changsheng, was from Jie County, Hedong.

At that time, he fled to Zhuojun. The First Lord gathered his troops in the village, with Guan Yu and Zhang Fei as his guards.

When the First Lord was the prime minister of Pingyuan, he appointed Guan Yu and Zhang Fei as commanders of other departments, commanding the troops respectively.

The late master and the two of them even slept in the same bed, like real brothers.

The two of them stood at the side of the late lord all day long in public, following the late lord everywhere without fear of danger. After the first lord attacked and killed Che Zhou, the governor of Xuzhou, he sent Guan Yu to guard Xiapi City, exercised his powers as prefect, and returned to Xiaopei himself.

In the fifth year of Jian'an, Cao Cao led his army to the east, and the First Lord defected to Yuan Shao.

After Cao Cao captured Guan Yu, he returned to the army and granted him the position of general, and treated him very politely.

Yuan Shao sent general Yan Liang to attack Liu Yan, the governor of Dongjun, at Baima. Cao Gong sent Zhang Liao and Guan Yu as the vanguard to attack him.

When Guan Yu saw the cover of Yan Liang's chariot, he rode on horseback to kill Yan Liang in full public view, beheading him, and then returned. Yuan Shao's generals were unable to resist him, and they broke the siege of the white horse.

Cao Cao immediately requested that Guan Yu be granted the title of Marquis Shouting of the Han Dynasty.

At first, Cao Cao thought that Guan Yu was very heroic, and after observing his mind, he didn't have any long-term plans to stay, so he said to Zhang Liao: You try to influence him with your emotions.

Soon, Zhang Liao wanted to question Guan Yu about this matter. Guan Yu sighed and said: I know that Duke Cao is very affectionate to me, but I am deeply loved by General Liu. I have vowed to live and die together and cannot betray him.

I will not stay after all, but I will serve and repay Duke Cao before leaving.

Zhang Liao reported Guan Yu's words to Cao Cao, and Cao Cao thought he was very righteous.

When Guan Yu killed Yan Liang, Cao Cao knew that he must leave, and he rewarded him very generously.

Guan Yu sealed up all the rewards he received, left a letter to say goodbye, and then went to Yuan Shao's army to join his former lord. Cao Cao's men wanted to chase him, but Cao Cao said: "Everyone is his own master, so stop chasing him."

Guan Yu followed Liu Bei and attached himself to Liu Biao.

After Liu Biao's death, Cao Cao pacified Jingzhou. The First Lord led his army from Fancheng to the south and crossed the river back. He also sent Guan Yu to take hundreds of ships to meet at Jiangling.

Cao Cao pursued them to Changban in Dangyang County. The First Lord took a small road and went straight to Hanjin. He happened to meet up with Guan Yu's fleet and arrived at Xiakou together.

Sun Quan sent troops to assist Liu Bei in fighting against Cao Cao. Cao Cao led his troops back. The First Lord occupied the counties south of the Yangtze River, rewarded and appointed meritorious officials, and appointed Guan Yu as the governor of Xiangyang and the general of the Dang bandits, stationed in the north of the Yangtze River.

"""

summary1 = """

The two of them stood at the side of the late lord all day long in public, following the late lord everywhere without fear of danger. After the first lord attacked and killed Che Zhou, the governor of Xuzhou, he sent Guan Yu to guard Xiapi City, exercised his powers as prefect, and returned to Xiaopei himself.

After Liu Biao's death, Cao Cao pacified Jingzhou. The First Lord led his army from Fancheng to the south and crossed the river back. He also sent Guan Yu to take hundreds of ships to meet at Jiangling.

Sun Quan sent troops to assist Liu Bei in fighting against Cao Cao. Cao Cao led his troops back. The First Lord occupied the counties south of the Yangtze River, rewarded and appointed meritorious officials, and appointed Guan Yu as the governor of Xiangyang and the general of the Dang bandits, stationed in the north of the Yangtze River.

"""


# Track time to estimate full dataset cost
start_time = time.time()

score_zs1 = model_zs.score([document], [summary1])
score_conv1 = model_conv.score([document], [summary1])

end_time = time.time()
elapsed_time = end_time - start_time

print("[Summary 1] SummaCZS Score: %.3f; SummacConv score: %.3f" % (score_zs1["scores"][0], score_conv1["scores"][0])) # [Summary 1] SummaCZS Score: 0.582; SummacConv score: 0.536
print(f"⏱️ Total time: {elapsed_time:.2f} seconds.")



In [ ]:
# --- Cell 5: Statistical Analysis & Visualization ---
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Use the exact output directory you defined in the previous cell
output_dir = "/content/drive/MyDrive/github/classical-chinese-summarization/data/processed/evaluation/summary_eval"
csv_path = f"{output_dir}/summac_test_results_fixed.csv"

def evaluate_summac_run(file_path):
    print(f"Loading results from: {file_path}")

    # Check if file exists before loading
    if not os.path.exists(file_path):
        print(f"Error: Could not find {file_path}. Did the previous cell finish running?")
        return None

    df = pd.read_csv(file_path)

    # Ensure the score column exists
    if 'summac_conv_score' not in df.columns:
        print("Error: 'summac_conv_score' column missing. Check your generation step.")
        return df

    # Clean the scores
    df['summac_conv_score'] = pd.to_numeric(df['summac_conv_score'], errors='coerce')
    scores = df['summac_conv_score'].dropna()

    # Calculate Statistics
    print("\n" + "="*40)
    print("📊 SUMMAC-CONV DESCRIPTIVE STATISTICS")
    print("="*40)
    print(scores.describe().round(4))

    """
    # Calculate Pass Rate (Faithfulness)
    pass_rate = (scores >= threshold).mean() * 100
    print("\n" + "="*40)
    print(f"🎯 FAITHFULNESS REPORT (Threshold: {threshold})")
    print("="*40)
    print(f"Total Summaries Evaluated : {len(scores)}")
    print(f"Faithful Summaries        : {pass_rate:.2f}%")
    print(f"Hallucinated Summaries    : {(100 - pass_rate):.2f}%")
    ###
    """

    # Generate the Histogram
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 6))

    # Plotting the distribution
    sns.histplot(scores, bins=25, kde=True, color='#4C72B0', edgecolor='black')

    # Threshold line
    #plt.axvline(x=threshold, color='#C44E52', linestyle='--', linewidth=2.5,
     #           label=f'Faithfulness Threshold ({threshold})')

    # Formatting for Academic Reporting
    plt.title('Distribution of SummaC-Conv Scores\n(English translation $\\rightarrow$ English Summaries)',
              fontsize=14, fontweight='bold')
    plt.xlabel('SummaC-Conv Entailment Score', fontsize=12)
    plt.ylabel('Frequency (Number of Documents)', fontsize=12)
    plt.legend(loc='upper right')
    plt.tight_layout()

    # Save the plot to the same Google Drive folder
    plot_path = f"{output_dir}/summac_distribution_plot.png"
    plt.savefig(plot_path, dpi=300)
    print(f"\n✅ High-resolution histogram saved to: {plot_path}")

    plt.show()

    return df

# Run the analysis
final_df = evaluate_summac_run(csv_path)

# Optional: View the worst-performing summaries to analyze hallucinations
if final_df is not None:
    print("\n🚨 Top 3 Most Hallucinated Summaries (Lowest Scores):")
    worst_summaries = final_df.sort_values(by='summac_conv_score').head(3)
    display(worst_summaries[['rel_path', 'summac_conv_score', 'summary']])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

output_dir = "/content/drive/MyDrive/github/classical-chinese-summarization/data/processed/evaluation/summary_eval"
csv_path = f"{output_dir}/summac_test_results_fixed.csv"

# Configure matplotlib to display Chinese characters (keeping this for title/metadata consistency if needed)
font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
if not fm.fontManager.findfont('WenQuanYi Zen Hei', fontext='ttf'):
    !apt-get update
    !apt-get install -y fonts-wqy-zenhei
    fm.fontManager.addfont(font_path)

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Load the fixed dataframe
df = pd.read_csv(csv_path)

# 1. Map Chinese corpus names to English translations
corpus_map = {
    '史记': 'Shiji (Records of the Grand Historian)',
    '汉书': 'Hanshu (Book of Han)',
    '后汉书': 'Houhanshu (Book of the Later Han)',
    '晋书': 'Jinshu (Book of Jin)',
    '梁书': 'Liangshu (Book of Liang)'
}

# Apply mapping - if a name isn't in the map, it keeps the original
df['corpus_en'] = df['corpus'].map(corpus_map).fillna(df['corpus'])

# Set up the plot style
sns.set_theme(style="ticks", palette="pastel")
plt.figure(figsize=(14, 8))

# Create a boxplot ordered by the median score of each corpus (using English names)
order = df.groupby('corpus_en')['summac_conv_score'].median().sort_values(ascending=False).index

sns.boxplot(x='corpus_en', y='summac_conv_score', data=df, order=order,
            width=0.6, fliersize=4, linewidth=1.5)

# Add a swarmplot to show the actual data points
sns.swarmplot(x='corpus_en', y='summac_conv_score', data=df, order=order,
              color=".25", size=3, alpha=0.6)

# Add the baseline threshold
#plt.axhline(y=0.45, color='red', linestyle='--', label='Suggested Faithfulness Threshold (0.45)')

plt.title('Factual Consistency by Historical Corpus\n(SummaC-Conv Entailment Scores)', fontsize=16, fontweight='bold')
plt.xlabel('Historical Text (Corpus)', fontsize=14)
plt.ylabel('Faithfulness Score', fontsize=14)
plt.xticks(rotation=15, ha='right')
plt.legend()
plt.tight_layout()

# Save for your presentation
plt.savefig('corpus_distribution_boxplot_en.png', dpi=300)
plt.show()